# Phase 2 — Construction d'un benchmark de dégradation documentaire

**Objectif** : générer des versions artificiellement dégradées des documents FUNSD (Phase 1) afin de tester, dans les phases suivantes, la robustesse de l'OCR et de l'extraction d'information face à des documents bruités.

8 dégradations implémentées, chacune sur 3 niveaux (`faible`, `moyen`, `fort`) :
1. Flou gaussien
2. Bruit aléatoire
3. Rotation légère
4. Faible contraste
5. Compression JPEG
6. Effet scan dégradé
7. Distorsion / décalage géométrique
8. Ombres / zones assombries

Le module `src/degradation.py` contient toute la logique. Ce notebook l'utilise, visualise les résultats et lance la génération complète du benchmark.

In [ ]:
import sys
import json
from pathlib import Path

sys.path.append("..")
from src.degradation import (
    apply_degradation,
    generate_degraded_dataset,
    DEGRADATIONS,
    LEVELS,
    PARAM_GRID,
)

from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path("..")
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "degraded"

manifest = json.loads((RAW_DIR / "manifest.json").read_text(encoding="utf-8"))
print(f"Documents disponibles (Phase 1) : {len(manifest)}")
print(f"Dégradations disponibles ({len(DEGRADATIONS)}) : {list(DEGRADATIONS.keys())}")
print(f"Niveaux : {LEVELS}")

## 1. Grille de paramètres

Récapitulatif des paramètres utilisés pour chaque dégradation, à chaque niveau de difficulté (documenté aussi dans `src/degradation.py`).

In [ ]:
import pandas as pd

rows = []
for deg, levels_params in PARAM_GRID.items():
    for level, params in levels_params.items():
        rows.append({"degradation": deg, "level": level, **params})

pd.DataFrame(rows)

## 2. Test qualitatif sur un document exemple

On applique les 8 dégradations (niveau `fort`, le plus visible) sur un même document, pour vérifier visuellement que chaque transformation est cohérente avant de lancer la génération sur tout le dataset.

In [ ]:
sample_entry = manifest[0]
sample_img = Image.open(RAW_DIR / "images" / sample_entry["image"]).convert("RGB")

fig, axes = plt.subplots(3, 3, figsize=(14, 16))
axes = axes.flatten()

axes[0].imshow(sample_img)
axes[0].set_title("original")
axes[0].axis("off")

for i, deg_name in enumerate(DEGRADATIONS.keys(), start=1):
    degraded_img, params = apply_degradation(sample_img, deg_name, level="fort", seed=42)
    axes[i].imshow(degraded_img)
    axes[i].set_title(deg_name)
    axes[i].axis("off")

plt.tight_layout()
out_fig = ROOT / "results" / "figures" / "phase2_degradations_apercu.png"
out_fig.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_fig, dpi=120, bbox_inches="tight")
plt.show()
print(f"Figure sauvegardée : {out_fig}")

## 3. Comparaison des 3 niveaux sur une même dégradation

Exemple avec le flou gaussien, pour illustrer la progression faible → moyen → fort.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(sample_img)
axes[0].set_title("original")
axes[0].axis("off")

for i, level in enumerate(LEVELS, start=1):
    degraded_img, params = apply_degradation(sample_img, "flou_gaussien", level=level, seed=42)
    axes[i].imshow(degraded_img)
    axes[i].set_title(f"flou_gaussien — {level}")
    axes[i].axis("off")

plt.tight_layout()
plt.savefig(ROOT / "results" / "figures" / "phase2_niveaux_flou.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Génération du benchmark complet

Application des 8 dégradations x 3 niveaux à **tous** les documents du dataset (train). Les images dégradées sont sauvegardées dans `data/degraded/images/`, et un manifest JSON (`data/degraded/manifest_degraded.json`) enregistre pour chaque image générée : le document d'origine, le type de dégradation, le niveau, la seed et les paramètres exacts utilisés (reproductibilité totale).

In [ ]:
manifest_degraded_path = generate_degraded_dataset(
    manifest_path=RAW_DIR / "manifest.json",
    raw_images_dir=RAW_DIR / "images",
    output_dir=OUT_DIR,
    seed=42,
)

## 5. Vérification et statistiques du benchmark généré

In [ ]:
records = json.loads(manifest_degraded_path.read_text(encoding="utf-8"))
df = pd.DataFrame(records)

print(f"Total images dégradées générées : {len(df)}")
print(f"Documents originaux couverts    : {df['document_original'].nunique()}")
print(f"Dégradations                    : {df['degradation'].nunique()}")
print(f"Niveaux                         : {df['level'].nunique()}")

display(df.groupby(["degradation", "level"]).size().unstack())

## 6. Note de synthèse (résultats attendus de la Phase 2)

- **Module de dégradation** : `src/degradation.py`, 8 fonctions paramétrables, 3 niveaux chacune, seed reproductible par (document, dégradation, niveau).
- **Benchmark généré** : `data/degraded/images/` (nom de fichier `<doc>__<degradation>__<level>.png`) + `data/degraded/manifest_degraded.json` recensant tous les paramètres utilisés.
- **Volume** : `nb_documents × 8 dégradations × 3 niveaux` images dégradées (voir tableau de statistiques ci-dessus pour les chiffres exacts sur le dataset complet).
- **Vérification qualitative** : les 8 dégradations produisent des effets visuellement distincts et progressifs (faible → fort), cohérents avec les cas réels de documents scannés dégradés décrits dans le cahier des charges.
- **Prochaine étape (Phase 3)** : appliquer un OCR brut (Tesseract / EasyOCR) sur les documents originaux **et** sur les documents dégradés, puis calculer CER/WER pour mesurer l'impact de chaque dégradation sur la reconnaissance de texte.